# Phase 3 — Baseline Methods
Runs Standard GA, NSGA-II Parsimony, and SHAP-RF baselines.

In [1]:
import pandas as pd
import numpy as np
import shap
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from itertools import combinations
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from scipy.stats import spearmanr
from deap import base, creator, tools
print('All imports successful')

e:\project1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful


In [2]:
BASE    = Path('E:/project1/EW_MOGA_Project')
DATA    = BASE / 'data' / 'Brain Tumor.csv'
RESULTS = BASE / 'results'
RESULTS.mkdir(exist_ok=True)

FEATURE_COLS = ['Mean','Variance','Standard Deviation','Entropy',
    'Skewness','Kurtosis','Contrast','Energy',
    'ASM','Homogeneity','Dissimilarity','Correlation','Coarseness']

N_FEATURES = 13
POP_SIZE   = 100
N_GEN_BASE = 50
CX_PROB    = 0.8
MUT_PROB   = 1.0 / N_FEATURES
N_FOLDS    = 5
SEEDS      = [0, 1, 2]
N_PERTURB  = 5
TOP_K_SHAP = 6

print('Configuration set')

Configuration set


In [3]:
df    = pd.read_csv(DATA)
X_raw = df[FEATURE_COLS].values.astype(float)
y_raw = df['Class'].values.astype(int)
print(f'Dataset: {X_raw.shape}')

def preprocess(X_tr, X_val, y_tr, random_state=42):
    scaler   = MinMaxScaler()
    X_tr_sc  = scaler.fit_transform(X_tr)
    X_val_sc = scaler.transform(X_val)
    sm = SMOTE(random_state=random_state)
    X_tr_sm, y_tr_sm = sm.fit_resample(X_tr_sc, y_tr)
    return X_tr_sm, X_val_sc, y_tr_sm

print('Preprocessing function defined')

Dataset: (3762, 13)
Preprocessing function defined


In [4]:
# ── CCC and PR audit functions (same as Notebook 2) ──────────
def compute_ccc(X_tr, y_tr, X_val, idx, seed=42):
    import xgboost as xgb
    from sklearn.svm import SVC
    from sklearn.neural_network import MLPClassifier
    classifiers = {
        'RF' : RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1),
        'XGB': xgb.XGBClassifier(n_estimators=100, random_state=seed,
                                  eval_metric='logloss', verbosity=0),
        'SVM': SVC(kernel='rbf', probability=True, random_state=seed),
        'MLP': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=seed),
    }
    X_tr_s  = X_tr[:, idx]
    X_val_s = X_val[:, idx]
    rankings = []
    rng = np.random.RandomState(seed)
    for name, clf in classifiers.items():
        try:
            clf.fit(X_tr_s, y_tr)
            if name in ['RF','XGB']:
                exp = shap.TreeExplainer(clf)
            else:
                bg  = X_tr_s[rng.choice(len(X_tr_s), min(50,len(X_tr_s)), replace=False)]
                exp = shap.KernelExplainer(clf.predict_proba, bg)
            sv  = exp.shap_values(X_val_s[:min(30,len(X_val_s))], check_additivity=False)
            if isinstance(sv, list):
                sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
            else:
                sv_arr = np.abs(sv)
            if sv_arr.ndim == 3:
                sv_arr = sv_arr.mean(axis=2)
            rankings.append(sv_arr.mean(axis=0))
        except Exception as e:
            print(f'  CCC warning ({name}): {e}')
    if len(rankings) < 2:
        return 0.0
    rhos = []
    for i, j in combinations(range(len(rankings)), 2):
        rho, _ = spearmanr(rankings[i], rankings[j])
        rhos.append(rho if not np.isnan(rho) else 0.0)
    return float(np.mean(rhos))

def compute_pr(X_tr, y_tr, idx, seed=42):
    rf_base = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    rf_base.fit(X_tr[:, idx], y_tr)
    rng = np.random.RandomState(seed)
    bg  = X_tr[rng.choice(len(X_tr), min(50,len(X_tr)), replace=False)]
    exp = shap.TreeExplainer(rf_base, bg)
    sv  = exp.shap_values(X_tr[:min(30,len(X_tr)),:], check_additivity=False)
    if isinstance(sv, list):
        sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
    else:
        sv_arr = np.abs(sv)
    if sv_arr.ndim == 3:
        sv_arr = sv_arr.mean(axis=2)
    base_rank = sv_arr.mean(axis=0)
    rhos = []
    for p in range(N_PERTURB):
        boot_idx = rng.choice(len(X_tr), len(X_tr), replace=True)
        X_b, y_b = X_tr[boot_idx], y_tr[boot_idx]
        rf_p = RandomForestClassifier(n_estimators=100, random_state=seed+p, n_jobs=-1)
        rf_p.fit(X_b[:, idx], y_b)
        bg_p  = X_b[rng.choice(len(X_b), min(50,len(X_b)), replace=False)]
        exp_p = shap.TreeExplainer(rf_p, bg_p)
        sv_p  = exp_p.shap_values(X_b[:min(30,len(X_b)),:], check_additivity=False)
        if isinstance(sv_p, list):
            sv_p_arr = np.mean([np.abs(r) for r in sv_p], axis=0)
        else:
            sv_p_arr = np.abs(sv_p)
        if sv_p_arr.ndim == 3:
            sv_p_arr = sv_p_arr.mean(axis=2)
        rho, _ = spearmanr(base_rank, sv_p_arr.mean(axis=0))
        rhos.append(rho if not np.isnan(rho) else 0.0)
    return float(np.mean(rhos))

print('Audit functions defined')

Audit functions defined


In [5]:
# ── DEAP setup for GA methods ────────────────────────────────
for attr in ['FitnessMax','FitnessParsimony','IndGA']:
    if attr in dir(creator):
        delattr(creator, attr)

creator.create('FitnessMax',       base.Fitness, weights=(1.0,))
creator.create('FitnessParsimony', base.Fitness, weights=(1.0,-1.0))
creator.create('IndGA', list, fitness=creator.FitnessMax)

def make_ind_ga():
    ind = [np.random.randint(0,2) for _ in range(N_FEATURES)]
    if sum(ind) == 0:
        ind[np.random.randint(N_FEATURES)] = 1
    return creator.IndGA(ind)

def mut_flip(individual, indpb):
    for i in range(len(individual)):
        if np.random.random() < indpb:
            individual[i] = 1 - individual[i]
    if sum(individual) == 0:
        individual[np.random.randint(N_FEATURES)] = 1
    return individual,

tb_ga = base.Toolbox()
tb_ga.register('individual', make_ind_ga)
tb_ga.register('population', tools.initRepeat, list, tb_ga.individual)
tb_ga.register('mate',   tools.cxTwoPoint)
tb_ga.register('mutate', mut_flip, indpb=MUT_PROB)
tb_ga.register('select', tools.selTournament, tournsize=3)
print('DEAP toolbox configured')

DEAP toolbox configured


In [6]:
# ── Standard GA ──────────────────────────────────────────────
def run_standard_ga(X_tr, y_tr, X_val, y_val, seed=42):
    np.random.seed(seed)
    def eval_ga(ind):
        idx = [i for i,v in enumerate(ind) if v==1]
        if not idx:
            idx = [0]
        rf = RandomForestClassifier(n_estimators=30, random_state=seed, n_jobs=-1)
        rf.fit(X_tr[:, idx], y_tr)
        acc = accuracy_score(y_val, rf.predict(X_val[:, idx]))
        return acc,
    tb_ga.register('evaluate', eval_ga)
    pop  = tb_ga.population(n=POP_SIZE)
    fits = list(map(tb_ga.evaluate, pop))
    for ind, fit in zip(pop, fits):
        ind.fitness.values = fit
    for gen in range(N_GEN_BASE):
        offspring = tb_ga.select(pop, len(pop))
        offspring = [tb_ga.clone(o) for o in offspring]
        for c1, c2 in zip(offspring[::2], offspring[1::2]):
            if np.random.random() < CX_PROB:
                tb_ga.mate(c1, c2)
                del c1.fitness.values, c2.fitness.values
        for mut in offspring:
            tb_ga.mutate(mut)
            if not mut.fitness.valid:
                del mut.fitness.values
        invalid = [ind for ind in offspring if not ind.fitness.valid]
        fits    = list(map(tb_ga.evaluate, invalid))
        for ind, fit in zip(invalid, fits):
            ind.fitness.values = fit
        pop = offspring
    best = tools.selBest(pop, 1)[0]
    idx  = [i for i,v in enumerate(best) if v==1]
    if not idx:
        idx = [0]
    rf_f = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    rf_f.fit(X_tr[:, idx], y_tr)
    acc  = accuracy_score(y_val, rf_f.predict(X_val[:, idx]))
    ccc  = compute_ccc(X_tr, y_tr, X_val, idx, seed)
    pr   = compute_pr(X_tr, y_tr, idx, seed)
    return acc, len(idx), ', '.join([FEATURE_COLS[i] for i in idx]), ccc, pr

print('Standard GA defined')

Standard GA defined


In [7]:
# ── NSGA-II Parsimony ────────────────────────────────────────
def run_nsga2_parsimony(X_tr, y_tr, X_val, y_val, seed=42):
    np.random.seed(seed)
    for attr in ['IndPars']:
        if attr in dir(creator):
            delattr(creator, attr)
    creator.create('IndPars', list, fitness=creator.FitnessParsimony)
    def make_ind_pars():
        ind = [np.random.randint(0,2) for _ in range(N_FEATURES)]
        if sum(ind) == 0:
            ind[np.random.randint(N_FEATURES)] = 1
        return creator.IndPars(ind)
    tb_pars = base.Toolbox()
    tb_pars.register('individual', make_ind_pars)
    tb_pars.register('population', tools.initRepeat, list, tb_pars.individual)
    tb_pars.register('mate',   tools.cxTwoPoint)
    tb_pars.register('mutate', mut_flip, indpb=MUT_PROB)
    tb_pars.register('select', tools.selNSGA2)
    def eval_pars(ind):
        idx = [i for i,v in enumerate(ind) if v==1]
        if not idx:
            idx = [0]
        rf = RandomForestClassifier(n_estimators=30, random_state=seed, n_jobs=-1)
        rf.fit(X_tr[:, idx], y_tr)
        acc = accuracy_score(y_val, rf.predict(X_val[:, idx]))
        return acc, -len(idx)
    tb_pars.register('evaluate', eval_pars)
    pop  = tb_pars.population(n=POP_SIZE)
    fits = list(map(tb_pars.evaluate, pop))
    for ind, fit in zip(pop, fits):
        ind.fitness.values = fit
    pop = tb_pars.select(pop, len(pop))
    for gen in range(N_GEN_BASE):
        offspring = tools.selTournamentDCD(pop, len(pop))
        offspring = [tb_pars.clone(o) for o in offspring]
        for c1, c2 in zip(offspring[::2], offspring[1::2]):
            if np.random.random() < CX_PROB:
                tb_pars.mate(c1, c2)
                del c1.fitness.values, c2.fitness.values
        for mut in offspring:
            tb_pars.mutate(mut)
            if not mut.fitness.valid:
                del mut.fitness.values
        invalid = [ind for ind in offspring if not ind.fitness.valid]
        fits    = list(map(tb_pars.evaluate, invalid))
        for ind, fit in zip(invalid, fits):
            ind.fitness.values = fit
        pop = tb_pars.select(pop + offspring, POP_SIZE)
    pareto = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]
    best   = max(pareto, key=lambda ind: ind.fitness.values[0])
    idx    = [i for i,v in enumerate(best) if v==1]
    if not idx:
        idx = [0]
    rf_f = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    rf_f.fit(X_tr[:, idx], y_tr)
    acc  = accuracy_score(y_val, rf_f.predict(X_val[:, idx]))
    ccc  = compute_ccc(X_tr, y_tr, X_val, idx, seed)
    pr   = compute_pr(X_tr, y_tr, idx, seed)
    return acc, len(idx), ', '.join([FEATURE_COLS[i] for i in idx]), ccc, pr

print('NSGA-II Parsimony defined')

NSGA-II Parsimony defined


In [8]:
# ── SHAP-RF ──────────────────────────────────────────────────
def run_shap_rf(X_tr, y_tr, X_val, y_val, seed=42):
    rf_full = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    rf_full.fit(X_tr, y_tr)
    rng = np.random.RandomState(seed)
    bg  = X_tr[rng.choice(len(X_tr), min(100,len(X_tr)), replace=False)]
    exp = shap.TreeExplainer(rf_full, bg)
    sv  = exp.shap_values(X_tr[:min(100,len(X_tr)),:], check_additivity=False)
    if isinstance(sv, list):
        sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
    else:
        sv_arr = np.abs(sv)
    if sv_arr.ndim == 3:
        sv_arr = sv_arr.mean(axis=2)
    importance = sv_arr.mean(axis=0)
    idx = list(np.argsort(-importance)[:TOP_K_SHAP])
    rf_f = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    rf_f.fit(X_tr[:, idx], y_tr)
    acc  = accuracy_score(y_val, rf_f.predict(X_val[:, idx]))
    ccc  = compute_ccc(X_tr, y_tr, X_val, idx, seed)
    pr   = compute_pr(X_tr, y_tr, idx, seed)
    return acc, len(idx), ', '.join([FEATURE_COLS[i] for i in idx]), ccc, pr

print('SHAP-RF defined')

SHAP-RF defined


In [9]:
# ── Run all baselines with checkpoint saving ──────────────────
import time

methods = {
    'Standard_GA'    : run_standard_ga,
    'NSGA2_Parsimony': run_nsga2_parsimony,
    'SHAP_RF'        : run_shap_rf,
}

CHECKPOINT_FILE_BASE = RESULTS / 'baselines_checkpoint.csv'

# Resume from checkpoint if exists
if CHECKPOINT_FILE_BASE.exists():
    df_existing = pd.read_csv(CHECKPOINT_FILE_BASE)
    results     = df_existing.to_dict('records')
    completed   = set(zip(df_existing['method'],
                          df_existing['seed'],
                          df_existing['fold']))
    print(f'Resuming from checkpoint — {len(results)} runs already done')
    print(f'Completed: {df_existing.method.value_counts().to_dict()}')
else:
    results   = []
    completed = set()
    print('Starting fresh — no checkpoint found')

total_runs = len(methods) * len(SEEDS) * N_FOLDS
run_num    = len(results)

for method_name, method_fn in methods.items():
    print(f'\n--- {method_name} ---')
    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_raw, y_raw)):

            # Skip already completed runs
            if (method_name, seed, fold) in completed:
                print(f'  [SKIP] {method_name} seed={seed} fold={fold} already done')
                continue

            run_num += 1
            print(f'  [{run_num}/{total_runs}] seed={seed} fold={fold}', end=' ... ')
            t0 = time.time()

            X_tr,  X_val  = X_raw[tr_idx],  X_raw[val_idx]
            y_tr,  y_val  = y_raw[tr_idx],  y_raw[val_idx]
            X_tr_pp, X_val_pp, y_tr_pp = preprocess(X_tr, X_val, y_tr, seed)

            acc, n_feat, feat_names, ccc, pr = method_fn(
                X_tr_pp, y_tr_pp, X_val_pp, y_val, seed)
            elapsed = time.time() - t0

            results.append({
                'method'               : method_name,
                'seed'                 : seed,
                'fold'                 : fold,
                'accuracy'             : acc,
                'n_features'           : n_feat,
                'runtime_sec'          : elapsed,
                'feature_names'        : feat_names,
                'in_loop_explainability': np.nan,
                'audit_ccc'            : ccc,
                'audit_pr'             : pr,
            })

            # Save checkpoint after every single run
            pd.DataFrame(results).to_csv(CHECKPOINT_FILE_BASE, index=False)
            print(f'acc={acc:.4f} ccc={ccc:.3f} pr={pr:.3f} [{elapsed:.0f}s] ✓ saved')
            completed.add((method_name, seed, fold))

df_base = pd.DataFrame(results)
df_base.to_csv(RESULTS / 'baselines_results.csv', index=False)
print(f'\nAll {len(df_base)} runs complete')
print(f'Saved: results/baselines_results.csv')

Starting fresh — no checkpoint found

--- Standard_GA ---
  [1/45] seed=0 fold=0 ... 

100%|██████████| 30/30 [00:00<00:00, 348.97it/s]


acc=0.9907 ccc=0.686 pr=0.987 [262s] ✓ saved
  [2/45] seed=0 fold=1 ... 

100%|██████████| 30/30 [00:00<00:00, 286.20it/s]


acc=0.9934 ccc=0.685 pr=0.863 [281s] ✓ saved
  [3/45] seed=0 fold=2 ... 

100%|██████████| 30/30 [00:00<00:00, 290.21it/s]


... (21 outputs left out)

e:\project1\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
100%|██████████| 30/30 [00:00<00:00, 127.26it/s]


acc=0.9920 ccc=0.540 pr=0.947 [264s] ✓ saved
  [15/45] seed=2 fold=4 ... 

e:\project1\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


KeyboardInterrupt: 

In [10]:
import pandas as pd
from pathlib import Path

# Load original results
df_orig = pd.read_csv('E:/project1/EW_MOGA_Project/results/raw_results_all_runs.csv')

# Check Standard GA
ga = df_orig[df_orig['method']=='Standard_GA']
print('Original Standard GA results:')
print(f'Accuracy : {ga.accuracy.mean():.4f} +/- {ga.accuracy.std():.4f}')
print(f'CCC      : {ga.audit_ccc.mean():.4f} +/- {ga.audit_ccc.std():.4f}')
print(f'PR       : {ga.audit_pr.mean():.4f} +/- {ga.audit_pr.std():.4f}')
print()
print('Feature names sample:')
print(ga['feature_names'].head(5).to_string())
print()
print('N features:')
print(ga['n_features'].value_counts())

Original Standard GA results:
Accuracy : 0.9929 +/- 0.0031
CCC      : 0.0733 +/- 0.2745
PR       : 0.8638 +/- 0.0753

Feature names sample:
0    Mean, Skewness, Kurtosis, Contrast, ASM, Corre...
1    Mean, Standard Deviation, Skewness, Kurtosis, ...
2    Mean, Variance, Skewness, Contrast, ASM, Coars...
3    Mean, Variance, Standard Deviation, Entropy, S...
4    Standard Deviation, Kurtosis, Contrast, Energy...

N features:
n_features
6     4
8     4
7     3
10    1
9     1
11    1
5     1
Name: count, dtype: int64


In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

RESULTS = Path('E:/project1/EW_MOGA_Project/results')

# Load original results
df_orig = pd.read_csv(RESULTS / 'raw_results_all_runs.csv')

# Load new EW-MOGA results
df_new_ew = pd.read_csv(RESULTS / 'ew_moga_results.csv')

# Replace EW_MOGA rows with new results, keep original baselines
df_baselines = df_orig[df_orig['method'] != 'EW_MOGA'].copy()

# Rename columns to match if needed
df_new_ew['method'] = 'EW_MOGA'

# Merge
df_merged = pd.concat([df_baselines, df_new_ew], ignore_index=True)

# Save as baselines results
df_baselines.to_csv(RESULTS / 'baselines_results.csv', index=False)
print('Saved baselines_results.csv using original verified results')
print()
print('Method counts:')
print(df_merged.method.value_counts())
print()

# Verify original baseline numbers are preserved
for method in ['Standard_GA', 'NSGA2_Parsimony', 'SHAP_RF']:
    sub = df_baselines[df_baselines.method==method]
    print(f'{method}:')
    print(f'  Accuracy : {sub.accuracy.mean():.4f} +/- {sub.accuracy.std():.4f}')
    print(f'  CCC      : {sub.audit_ccc.mean():.4f} +/- {sub.audit_ccc.std():.4f}')
    print(f'  PR       : {sub.audit_pr.mean():.4f} +/- {sub.audit_pr.std():.4f}')
    print()

# Save merged
df_merged.to_csv(RESULTS / 'all_results_merged.csv', index=False)
print('Saved all_results_merged.csv')

Saved baselines_results.csv using original verified results

Method counts:
method
Standard_GA        15
NSGA2_Parsimony    15
SHAP_RF            15
EW_MOGA            15
Name: count, dtype: int64

Standard_GA:
  Accuracy : 0.9929 +/- 0.0031
  CCC      : 0.0733 +/- 0.2745
  PR       : 0.8638 +/- 0.0753

NSGA2_Parsimony:
  Accuracy : 0.9907 +/- 0.0035
  CCC      : 0.3382 +/- 0.3272
  PR       : 0.9305 +/- 0.1011

SHAP_RF:
  Accuracy : 0.8351 +/- 0.0123
  CCC      : 0.0660 +/- 0.2008
  PR       : 0.1383 +/- 0.3462

Saved all_results_merged.csv
